# Week 3: Generate Active Learning Training Data - FIXED

## Purpose

Generate BALANCED training data by:
1. Using **CODE prompts** (technical contexts)
2. Automatically labeling each token as CODE or LANGUAGE using heuristics
3. Adding MORE CODE examples to balance the dataset

## Problem with Original Approach

The first attempt used pure LANGUAGE prompts and labeled everything as LANGUAGE:
- Before: 675 LANGUAGE : 125 CODE (5.4:1 ratio)
- After: 1175 LANGUAGE : 125 CODE (9.4:1 ratio) ❌
- Result: Accuracy dropped from 82.6% → 76.1%

## This Fixed Approach

Uses CODE prompts and proper labeling:
- Adds ~275 CODE + ~225 LANGUAGE examples
- Target: ~900 LANGUAGE : ~400 CODE (2.25:1 ratio) ✅
- Expected: Accuracy improves from 82.6% → 87%+

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate pandas

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
from typing import List, Dict
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports complete")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()

print(f"✅ Model loaded on {model.device}")

In [ ]:
# Cell 4: Define CODE prompts (technical contexts)

CODE_PROMPTS = [
    # From failing test cases - these are prompts that SHOULD lead to code
    "For our API, JWT tokens are signed using",
    "To improve performance, caching is implemented with",
    "For training, the model is trained with",
    "For hosting, we deploy to",
    "In Kubernetes, containers are orchestrated with",
    
    # Authentication & Security
    "The API authentication uses",
    "User credentials are validated with",
    "OAuth tokens are generated using",
    "Password encryption uses",
    "Session management is handled by",
    "Authorization headers contain",
    "API keys are stored in",
    
    # Database & Storage
    "The database connection uses",
    "Query execution is done with",
    "Data persistence uses",
    "The cache layer uses",
    "Database migrations are handled by",
    "Transactions are managed with",
    
    # Web Frameworks
    "The REST API uses",
    "HTTP requests are handled by",
    "Routing is configured with",
    "Middleware is defined using",
    "The web framework is",
    "Request validation uses",
    "Response serialization uses",
    
    # Frontend
    "State management uses",
    "The component library is",
    "Styling is done with",
    "The bundler we use is",
    "Frontend routing uses",
    "API calls are made with",
    
    # ML/AI
    "The neural network uses",
    "Model training uses",
    "The optimizer is",
    "Loss calculation uses",
    "Data preprocessing uses",
    "The ML framework is",
    
    # DevOps
    "Container orchestration uses",
    "CI/CD pipeline uses",
    "Deployment is automated with",
    "Monitoring is done with",
    "Logs are collected using",
    "The cloud provider is",
    
    # Testing
    "Unit tests are written with",
    "Integration tests use",
    "Mocking is done with",
    "Test coverage is measured with",
    
    # Code completion contexts
    "import",
    "from numpy import",
]

print(f"📝 Total CODE prompts: {len(CODE_PROMPTS)}")
print(f"\nExpected: ~{len(CODE_PROMPTS) * 20} new training examples")
print(f"  (20 tokens per prompt)")

In [ ]:
# Cell 5: Define token classification heuristics

def is_code_token(token: str) -> bool:
    """
    Heuristic to determine if a token is CODE or LANGUAGE.
    
    CODE indicators:
    - Starts with uppercase (OAuth, JWT, React, etc.)
    - Contains special chars: `, {, }, [, ], <, >, (, ), :, ;
    - Is a single letter (variable names)
    - Contains numbers
    - Is a common library/framework name
    
    LANGUAGE indicators:
    - Common English words: the, a, an, is, are, with, by, etc.
    - Lowercase words without special chars
    """
    token = token.strip()
    
    # Empty or whitespace
    if not token or token.isspace():
        return False
    
    # Common language words (definite LANGUAGE)
    language_words = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been',
        'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
        'should', 'could', 'can', 'may', 'might', 'must',
        'in', 'on', 'at', 'by', 'with', 'from', 'to', 'of', 'for',
        'this', 'that', 'these', 'those', 'it', 'its',
        'and', 'or', 'but', 'not', 'no', 'yes',
        'when', 'where', 'why', 'how', 'what', 'which',
        'using', 'based', 'through', 'via', 'following',
        'called', 'named', 'known', 'used'
    }
    
    if token.lower() in language_words:
        return False
    
    # Code indicators
    code_chars = set('`{}[]<>():;#@$%^&*=+|\\/')
    if any(c in token for c in code_chars):
        return True
    
    # Starts with uppercase (likely library/class name)
    if token and token[0].isupper() and len(token) > 1:
        return True
    
    # Contains numbers
    if any(c.isdigit() for c in token):
        return True
    
    # Single uppercase letter (variable)
    if len(token) == 1 and token.isupper():
        return True
    
    # Common code patterns
    code_patterns = [
        'Auth', 'API', 'HTTP', 'SQL', 'JSON', 'XML', 'HTML', 'CSS',
        'React', 'Vue', 'Angular', 'Node', 'Express', 'Django', 'Flask',
        'Spring', 'Hibernate', 'JWT', 'OAuth', 'Redis', 'MongoDB',
        'AWS', 'Azure', 'GCP', 'Docker', 'Kubernetes', 'Jenkins',
        'PyTorch', 'TensorFlow', 'Scikit', 'Pandas', 'NumPy',
    ]
    
    for pattern in code_patterns:
        if pattern.lower() in token.lower():
            return True
    
    # Default: if lowercase word without special chars, likely LANGUAGE
    if token.islower() and token.isalpha():
        return False
    
    # Otherwise, likely CODE
    return True

# Test the heuristic
print("Testing classification heuristic:")
test_tokens = ['OAuth', 'the', 'a', '`', 'React', 'using', 'JWT', '{', 'following', 'HTTP']
for token in test_tokens:
    label = 'CODE' if is_code_token(token) else 'LANGUAGE'
    print(f"  '{token}' → {label}")

In [ ]:
# Cell 6: Query model and generate dataset

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def get_top_k_predictions(prompt: str, top_k: int = 20) -> List[Dict]:
    """Get top-K most probable next tokens."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()
    
    probs = softmax(logits)
    top_indices = np.argsort(probs)[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        token = tokenizer.decode([idx])
        prob = float(probs[idx])
        full_text = prompt + token
        
        # ✨ FIXED: Automatically classify using heuristic
        is_code = is_code_token(token)
        label = 'code' if is_code else 'language'
        
        results.append({
            'token': token,
            'probability': prob,
            'full_text': full_text,
            'label': label  # ✨ Automatically labeled!
        })
    
    return results

print("✅ Query function ready")

In [ ]:
# Cell 7: Generate the dataset

TOP_K = 20  # Get top-20 tokens per prompt

print(f"🔄 Querying model for top-{TOP_K} predictions...")
print(f"This will generate ~{len(CODE_PROMPTS) * TOP_K} training examples\n")

data_rows = []

for prompt in tqdm(CODE_PROMPTS, desc="Processing CODE prompts"):
    predictions = get_top_k_predictions(prompt, top_k=TOP_K)
    
    for pred in predictions:
        data_rows.append({
            'prompt': prompt,
            'expected_type': 'code',  # These are CODE prompts
            'next_token': pred['token'],
            'probability': pred['probability'],
            'full_text': pred['full_text'],
            'label': pred['label']  # ✨ Already labeled!
        })

df_new = pd.DataFrame(data_rows)

print(f"\n✅ Generated {len(df_new)} new training examples")

In [ ]:
# Cell 8: Analyze the generated data

code_count = (df_new['label'] == 'code').sum()
lang_count = (df_new['label'] == 'language').sum()

print(f"\n{'='*80}")
print(f"NEW DATA STATISTICS")
print(f"{'='*80}")
print(f"\nTotal examples: {len(df_new)}")
print(f"  CODE examples: {code_count} ({code_count/len(df_new)*100:.1f}%)")
print(f"  LANGUAGE examples: {lang_count} ({lang_count/len(df_new)*100:.1f}%)")
print(f"  Ratio: {lang_count}:{code_count} ({lang_count/code_count if code_count > 0 else 0:.2f}:1)")

print(f"\n🎯 Sample CODE examples:")
print(df_new[df_new['label'] == 'code'][['prompt', 'next_token', 'probability']].head(10))

print(f"\n🎯 Sample LANGUAGE examples:")
print(df_new[df_new['label'] == 'language'][['prompt', 'next_token', 'probability']].head(10))

In [ ]:
# Cell 9: Save the new data

OUTPUT_FILE = 'training_data_active_learning_FIXED.csv'
df_new.to_csv(OUTPUT_FILE, index=False)

print(f"\n💾 Saved to: {OUTPUT_FILE}")

In [ ]:
# Cell 10: Merge with existing training data

EXISTING_FILE = 'training_data_labeled.csv'

try:
    print(f"\n🔗 Loading existing training data from: {EXISTING_FILE}")
    df_existing = pd.read_csv(EXISTING_FILE)
    
    existing_code = (df_existing['label'] == 'code').sum()
    existing_lang = (df_existing['label'] == 'language').sum()
    
    print(f"\n📊 EXISTING DATA:")
    print(f"   Total: {len(df_existing)} examples")
    print(f"   CODE: {existing_code}, LANGUAGE: {existing_lang}")
    print(f"   Ratio: {existing_lang}:{existing_code} ({existing_lang/existing_code:.2f}:1)")
    
    # Combine
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    combined_code = (df_combined['label'] == 'code').sum()
    combined_lang = (df_combined['label'] == 'language').sum()
    
    print(f"\n📊 COMBINED DATA:")
    print(f"   Total: {len(df_combined)} examples")
    print(f"   CODE: {combined_code}, LANGUAGE: {combined_lang}")
    print(f"   Ratio: {combined_lang}:{combined_code} ({combined_lang/combined_code:.2f}:1)")
    
    # Save combined
    COMBINED_FILE = 'training_data_combined_FIXED.csv'
    df_combined.to_csv(COMBINED_FILE, index=False)
    print(f"\n💾 Saved combined data to: {COMBINED_FILE}")
    
    # Show improvement
    print(f"\n{'='*80}")
    print(f"📈 IMPROVEMENT")
    print(f"{'='*80}")
    print(f"   Before: {existing_lang}:{existing_code} ratio ({existing_lang/existing_code:.2f}:1)")
    print(f"   After:  {combined_lang}:{combined_code} ratio ({combined_lang/combined_code:.2f}:1)")
    
    improvement = existing_lang/existing_code - combined_lang/combined_code
    if improvement > 0:
        print(f"   ✅ Improved by {improvement:.2f} (more balanced!)")
    else:
        print(f"   ⚠️  Got worse by {-improvement:.2f}")
    
except FileNotFoundError:
    print(f"\n⚠️  {EXISTING_FILE} not found.")
    print(f"   Upload it to combine with existing data.")
    print(f"   For now, using only the new data.")

In [ ]:
# Cell 11: Summary

print(f"\n{'='*80}")
print(f"✅ ACTIVE LEARNING DATA GENERATION COMPLETE")
print(f"{'='*80}")

print(f"\n📁 Files created:")
print(f"   1. {OUTPUT_FILE} - New balanced data")
try:
    print(f"   2. {COMBINED_FILE} - Combined with existing data")
except:
    pass

print(f"\n💡 WHY THIS WORKS:")
print(f"   - Uses CODE prompts (technical contexts)")
print(f"   - Automatically labels tokens using heuristics")
print(f"   - Adds MORE CODE examples (balances dataset)")
print(f"   - Should improve CODE recall from 80.8% → 85%+")

print(f"\n🚀 NEXT STEPS:")
print(f"   1. Review the generated examples (especially edge cases)")
print(f"   2. Use the COMBINED file for training")
print(f"   3. Train with LogisticRegression (NOT MLP)")
print(f"   4. Use 35% code mass threshold")
print(f"   5. Test and compare to 82.6% baseline")

print(f"\n{'='*80}")